# Tutorial 02 — Gobernanza en `alexandria-server` (`/v1`) y Distribución Inmutable (`/pub`)

En este notebook levantaremos una instancia real de **`alexandria-server`** (`C-HUB`) y recorreremos mediante **peticiones HTTP** todo el ciclo de vida de gobernanza:
1. Resolución del documento **W3C `did:web`** (`GET /.well-known/did.json`).
2. Autenticación Bearer y matriz de autorización por rol (`editor`, `reviewer`, `releaser`, `member`).
3. Creación y edición de **borradores (`Drafts`)** con control de concurrencia optimista (`ETag` / `If-Match`).
4. Propuesta de un **candidato inmutable (`:propose`)** que compila el YAML `eunomia.profiles/v1` al vuelo.
5. Separación de responsabilidades y aprobación firmada de una **Release (`:approve`)**.
6. Consulta del listado firmado en **JCS + Ed25519 (`GET /v1/versions`)** y descarga de artefactos inmutables direccionados por contenido en **`/pub/*`**.

In [1]:
from lab import AlexandriaLab, http_request

lab = AlexandriaLab(compiler="dsl", initial_clock="2026-09-01T10:00:00Z")
lab.start_server()

✔ alexandria-server listening at http://127.0.0.1:60380 (DID: did:web:127.0.0.1%3A60380)

## 1. Descubrimiento de Identidad: `GET /.well-known/did.json`

Al arrancar, `alexandria-server` genera su par de claves Ed25519 y publica su método de verificación en `/.well-known/did.json` (`REQ-HUB-010`, `DP-4`). Este endpoint es público:

In [2]:
status, headers, did_doc = http_request("GET", f"{lab.hub_url}/.well-known/did.json")
assert status == 200
assert did_doc["id"] == lab.hub_did

cache-control,no-cache
connection,close
content-length,403
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


## 2. Autenticación y Control de Acceso en `/v1/*` (`REQ-HUB-017`)

Todos los recursos bajo `/v1/` exigen un token Bearer válido (`Authorization: Bearer <token>`):
- Si no enviamos token, el servidor responde `401 Unauthorized` con `WWW-Authenticate: Bearer realm="vhub"` y código `API_FORBIDDEN_ROLE`.
- Si enviamos un token con rol insuficiente (`member-token` intentando crear un borrador), responde `403 Forbidden` (`API_FORBIDDEN_ROLE`).
- Con `member-token` en `GET /v1/versions` antes de publicar la primera versión, la autenticación pasa y devuelve `404 API_NOT_FOUND` (`"no version has been published yet"`).

In [3]:
print("1) Petición sin credenciales (esperado 401):")
st_unauth, _, _ = http_request("GET", f"{lab.hub_url}/v1/versions")
assert st_unauth == 401

print("2) Petición con rol insuficiente: member intentando crear un draft (esperado 403):")
st_forb, _, _ = http_request("POST", f"{lab.hub_url}/v1/drafts", token="member-token", body="dsl: eunomia.profiles/v1")
assert st_forb == 403

print("3) Petición autenticada con member-token antes de publicar versiones (esperado 404 API_NOT_FOUND):")
st_empty, _, _ = http_request("GET", f"{lab.hub_url}/v1/versions", token="member-token")
assert st_empty == 404

1) Petición sin credenciales (esperado 401):
2) Petición con rol insuficiente: member intentando crear un draft (esperado 403):
3) Petición autenticada con member-token antes de publicar versiones (esperado 404 API_NOT_FOUND):


connection,close
content-length,178
content-type,application/problem+json
date,"Tue, 01 Sep 2026 10:00:00 GMT"
www-authenticate,"Bearer realm=""vhub"""


connection,close
content-length,195
content-type,application/problem+json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


connection,close
content-length,173
content-type,application/problem+json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


## 3. Creación de Borradores (`Drafts`) y Concurrencia Optimista (`REQ-HUB-018`)

Como **`editor`** (`editor-token`), creamos un borrador enviando el bundle YAML `v1.0.0.yaml` a `POST /v1/drafts`.
Después probamos el control de concurrencia optimista en `PUT /v1/drafts/{id}`:
1. Sin cabecera `If-Match` -> **`428 Precondition Required` (`API_PRECONDITION_REQUIRED`)**.
2. Con `If-Match: "rev-1"` -> **`200 OK`** (avanza a `"rev-2"`).
3. Reutilizando el ETag antiguo `"rev-1"` -> **`412 Precondition Failed` (`API_PRECONDITION_FAILED`)**.

In [4]:
draft_yaml = (lab.root / "conformance/f6/drafts/v1.0.0.yaml").read_text(encoding="utf-8")

print("1) Crear borrador (POST /v1/drafts):")
st, hdrs, draft_resp = http_request("POST", f"{lab.hub_url}/v1/drafts", token="editor-token", body=draft_yaml)
assert st == 201
draft_id = draft_resp["id"]
etag_v1 = hdrs["etag"]

print("2) Intentar PUT sin If-Match (esperado 428 API_PRECONDITION_REQUIRED):")
st_no_if, _, _ = http_request("PUT", f"{lab.hub_url}/v1/drafts/{draft_id}", token="editor-token", body=draft_yaml)
assert st_no_if == 428

print("3) Reemplazar borrador con If-Match válido (esperado 200 OK):")
st_put, hdrs_put, _ = http_request(
    "PUT",
    f"{lab.hub_url}/v1/drafts/{draft_id}",
    token="editor-token",
    headers={"If-Match": etag_v1},
    body=draft_yaml,
)
assert st_put == 200
etag_v2 = hdrs_put["etag"]

print("4) Intentar PUT con ETag obsoleto rev-1 (esperado 412 API_PRECONDITION_FAILED):")
st_stale, _, _ = http_request(
    "PUT",
    f"{lab.hub_url}/v1/drafts/{draft_id}",
    token="editor-token",
    headers={"If-Match": etag_v1},
    body=draft_yaml,
)
assert st_stale == 412

1) Crear borrador (POST /v1/drafts):
2) Intentar PUT sin If-Match (esperado 428 API_PRECONDITION_REQUIRED):
3) Reemplazar borrador con If-Match válido (esperado 200 OK):
4) Intentar PUT con ETag obsoleto rev-1 (esperado 412 API_PRECONDITION_FAILED):


connection,close
content-length,33
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"
etag,"""40e7b8ac5f35eba5-1"""
location,/v1/drafts/40e7b8ac5f35eba5


connection,close
content-length,197
content-type,application/problem+json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


connection,close
content-length,33
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"
etag,"""40e7b8ac5f35eba5-2"""


connection,close
content-length,239
content-type,application/problem+json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


## 4. Validar (`:validate`) y Proponer un Candidato (`:propose`, `REQ-HUB-002`)

Antes de proponer, el editor puede ejecutar `POST /v1/drafts/{id}:validate`.
Al llamar a `POST /v1/drafts/{id}:propose`, el Hub compila el YAML con `compiler_dsl`, almacena todos los artefactos en el `BlobStore` direccionado por SHA-256 y congela un **Candidato inmutable**:

In [5]:
print("1) Validar borrador (POST /v1/drafts/{id}:validate):")
st_val, _, _ = http_request("POST", f"{lab.hub_url}/v1/drafts/{draft_id}:validate", token="editor-token")
assert st_val == 200

print("2) Proponer candidato (POST /v1/drafts/{id}:propose):")
st_prop, _, cand = http_request("POST", f"{lab.hub_url}/v1/drafts/{draft_id}:propose", token="editor-token")
assert st_prop == 201
cand_id = cand["id"]

print("3) Consultar candidato congelado (GET /v1/candidates/{id}):")
st_get_c, _, _ = http_request("GET", f"{lab.hub_url}/v1/candidates/{cand_id}", token="releaser-token")
assert st_get_c == 200

1) Validar borrador (POST /v1/drafts/{id}:validate):
2) Proponer candidato (POST /v1/drafts/{id}:propose):
3) Consultar candidato congelado (GET /v1/candidates/{id}):


connection,close
content-length,46
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


connection,close
content-length,1103
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


connection,close
content-length,1103
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


## 5. Separación de Roles y Aprobación de Release (`:approve`, `REQ-HUB-003`, `REQ-HUB-005`)

La especificación exige que un `editor` **no pueda aprobar** un candidato (`REQ-HUB-003`): solo el rol `authority` (`releaser`) puede ejecutar `:approve`.
Además, una vez aprobado el candidato, intentar aprobarlo una segunda vez devuelve `409 Conflict` (`API_INVALID_STATE`, `REQ-HUB-002`).

In [6]:
print("1) Un editor intenta aprobar el candidato (esperado 403 API_FORBIDDEN_ROLE):")
st_forb, _, _ = http_request("POST", f"{lab.hub_url}/v1/candidates/{cand_id}:approve", token="editor-token", body={})
assert st_forb == 403

print("2) El releaser aprueba el candidato (esperado 201 Created con sobre firmado eunomia.version/v1):")
st_app, _, release_rec = http_request("POST", f"{lab.hub_url}/v1/candidates/{cand_id}:approve", token="releaser-token", body={})
assert st_app == 201

print("3) Intentar aprobar de nuevo el mismo candidato (esperado 409 API_INVALID_STATE):")
st_dup, _, _ = http_request("POST", f"{lab.hub_url}/v1/candidates/{cand_id}:approve", token="releaser-token", body={})
assert st_dup == 409

1) Un editor intenta aprobar el candidato (esperado 403 API_FORBIDDEN_ROLE):
2) El releaser aprueba el candidato (esperado 201 Created con sobre firmado eunomia.version/v1):
3) Intentar aprobar de nuevo el mismo candidato (esperado 409 API_INVALID_STATE):


connection,close
content-length,214
content-type,application/problem+json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


connection,close
content-length,555
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


connection,close
content-length,190
content-type,application/problem+json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


## 6. Listado Firmado (`/v1/versions`) y Distribución Inmutable (`/pub/*`)

Una vez aprobada la versión `1.0.0`:
- `GET /v1/versions` devuelve el listado firmado `eunomia.versions/v1` con `Cache-Control: private, no-cache` (`REQ-HUB-012`).
- `GET /pub/demo` responde `303 See Other` redirigiendo al manifiesto de la versión cabeza (`REQ-HUB-016`, `DP-15`).
- `GET /pub/packages/1.0.0/manifest.json`, `space-lock.json` y las rutas versionadas `/pub/demo/1.0.0/...` se sirven sin autenticación con `Cache-Control: public, max-age=31536000, immutable` y `ETag: "<sha256>"` (`REQ-SYS-005`, `REQ-HUB-011`).

In [7]:
print("1) Listado firmado GET /v1/versions:")
st_list, hdrs_list, listing = http_request("GET", f"{lab.hub_url}/v1/versions", token="member-token")
assert st_list == 200
assert "private, no-cache" in hdrs_list.get("cache-control", "")

print("2) Avanzamos el reloj tras la ventana de propagación P2D (2026-09-03T12:00:00Z) y consultamos GET /pub/demo (303 See Other):")
lab.set_clock("2026-09-03T12:00:00Z")
st_red, hdrs_red, _ = http_request("GET", f"{lab.hub_url}/pub/demo", follow_redirects=False)
assert st_red == 303

print("3) Manifiesto firmado del paquete GET /pub/demo/1.0.0/manifest.json (y por digest en /pub/packages/{digest}/manifest.json):")
st_man, hdrs_man, manifest = http_request("GET", f"{lab.hub_url}/pub/demo/1.0.0/manifest.json")
assert st_man == 200
assert "immutable" in hdrs_man.get("cache-control", "")

print("4) Descarga directa de un artefacto compilado (JSON Schema de políticas):")
st_sch, _, _ = http_request("GET", f"{lab.hub_url}/pub/demo/1.0.0/profiles/odrl/f6/1.0.0/schema.json")
assert st_sch == 200

1) Listado firmado GET /v1/versions:
2) Avanzamos el reloj tras la ventana de propagación P2D (2026-09-03T12:00:00Z) y consultamos GET /pub/demo (303 See Other):
3) Manifiesto firmado del paquete GET /pub/demo/1.0.0/manifest.json (y por digest en /pub/packages/{digest}/manifest.json):
4) Descarga directa de un artefacto compilado (JSON Schema de políticas):


cache-control,"private, no-cache"
connection,close
content-length,609
content-type,application/json
date,"Tue, 01 Sep 2026 10:00:00 GMT"


🕒 Clock updated → 2026-09-03T12:00:00Z

cache-control,no-cache
connection,close
content-length,0
date,"Thu, 03 Sep 2026 12:00:00 GMT"
location,/pub/demo/1.0.0/manifest.json


cache-control,"public, max-age=31536000, immutable"
connection,close
content-length,2032
content-type,application/json
date,"Thu, 03 Sep 2026 12:00:00 GMT"
etag,"""9109e5b8ca1178902795040ad016be38a6a978c4c6fbb6e21492ac5e78f6e3ec"""


cache-control,"public, max-age=31536000, immutable"
connection,close
content-type,application/schema+json
date,"Thu, 03 Sep 2026 12:00:00 GMT"
etag,"""764b9f692a51e18ff079adf26fb8c902dc3caf5d529e30f033a579b478f4de80"""
transfer-encoding,chunked


## 7. Registro Inmutable de Eventos de Gobernanza (`GET /v1/events`)

Cada mutación en el Hub queda registrada secuencialmente en la tabla inmutable `events` de SQLite (`REQ-HUB-020`):

In [8]:
st_ev, _, events_resp = http_request("GET", f"{lab.hub_url}/v1/events?after=0", token="editor-token")
assert st_ev == 200
lab.close()

connection,close
content-length,685
content-type,application/json
date,"Thu, 03 Sep 2026 12:00:00 GMT"
